# The postdoc problem: select the second-best candidate

Candidates arrive in a **uniformly random order**. Absolute rank $1$ is best, and the goal is to select absolute rank $2$. We observe only the relative rank among candidates seen so far.

Unlike the classical best-candidate problem, a relative-rank-2 observation may be worth selecting. All decisions are irrevocable, and at most one candidate can be selected.


## Exact backward induction

If the current candidate arrives at position $k$ and has relative rank $a$, then their conditional probability of being absolute rank $2$ is

$$
g_{n,k}(a)=
\begin{cases}
\frac{k(n-k)}{n(n-1)}, & a=1,\\
\frac{k(k-1)}{n(n-1)}, & a=2,\\
0, & a\ge 3.
\end{cases}
$$

Let $V_k$ denote the optimal win probability **before observing** candidate $k$. With $V_{n+1}=0$,

$$
V_k=\frac{1}{k}\sum_{a=1}^k\max\{g_{n,k}(a),V_{k+1}\}.
$$

At time $k$, stop if $g_{n,k}(a)\ge V_{k+1}$ and the stopping reward is positive. Otherwise continue (including ties at zero). This is an exact finite-horizon solution.


In [ ]:
from fractions import Fraction
import random

def reward(n, k, a):
    if n < 2 or not (1 <= k <= n and 1 <= a <= k):
        raise ValueError("Require n >= 2, 1 <= k <= n, 1 <= a <= k.")
    if a == 1:
        return Fraction(k*(n-k), n*(n-1))
    if a == 2:
        return Fraction(k*(k-1), n*(n-1))
    return Fraction(0)

def backward_values(n):
    if n < 2:
        raise ValueError("n must be >= 2")
    values = [Fraction(0)] * (n+2)
    for k in range(n, 0, -1):
        values[k] = sum(
            (max(reward(n,k,a), values[k+1]) for a in range(1,k+1)),
            Fraction(0)
        ) / k
    return values

def should_stop(n,k,a,values):
    return reward(n,k,a)>0 and reward(n,k,a)>=values[k+1]

n = 10
values = backward_values(n)
print(f"Exact optimal success probability for n={n}: {values[1]} = {float(values[1]):.4%}")
for k in range(1,n+1):
    choices = [a for a in range(1,k+1) if should_stop(n,k,a,values)]
    print(f"Position {k}: accept relative ranks {choices or 'none'}")
assert backward_values(2)[1] == Fraction(1,2)


## Monte Carlo simulation of the optimal rule

A uniformly random permutation represents absolute ranks. At each position we compute only the candidate's relative rank from previously seen candidates. The optimal policy is applied without access to the hidden absolute rank. We reveal it after stopping, to decide whether the selection is a success.

If no candidate is selected, the trial is counted as a failure.


In [ ]:
def simulate_postdoc(n, trials=20000, seed=2026):
    if trials <= 0:
        raise ValueError("trials must be positive")
    v = backward_values(n)
    rng = random.Random(seed)
    wins = 0
    for _ in range(trials):
        order = list(range(1,n+1))
        rng.shuffle(order)
        observed = []
        for k, rank in enumerate(order, 1):
            a = 1 + sum(earlier < rank for earlier in observed)
            observed.append(rank)
            if should_stop(n,k,a,v):
                wins += rank == 2
                break
    return wins / trials

observed = simulate_postdoc(n)
print(f"Monte Carlo success: {observed:.4%}")
print(f"Exact optimal success: {float(values[1]):.4%}")
print(f"Difference: {abs(observed-float(values[1])):.4%}")


## Compare sample sizes

The optimal success probability varies with the horizon, and the optimal policy is not assumed to have the classical single-threshold form.


In [ ]:
sizes = [3,5,10,20,40]
theory = [float(backward_values(size)[1]) for size in sizes]
simulation = [simulate_postdoc(size, trials=5000, seed=2026) for size in sizes]
for size,t,est in zip(sizes,theory,simulation):
    print(f"n={size:2d}: theory={t:.4%}, simulation={est:.4%}")
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    plt.plot(sizes, theory, marker="o", label="Exact optimal value")
    plt.scatter(sizes, simulation, label="Monte Carlo")
    plt.xlabel("Number of candidates")
    plt.ylabel("Probability of selecting absolute rank 2")
    plt.legend()
    plt.show()
except ImportError:
    print("Install matplotlib to view the optional figure.")


## Exercises

1. Verify the conditional stopping rewards for $n=5$, $k=3$.
2. Use the recursion to compute $V_1$ exactly for $n=4$.
3. Compare the optimal postdoc policy with a rule that selects only relative-rank-1 candidates.
4. Explain why a relative rank of $3$ or larger cannot correspond to the overall second-best candidate.

**Interpretation:** the mathematical backward recursion determines the policy; simulation independently illustrates its success rate.
